# 03 - CWRU Baseline Fault Classification

Train a Random Forest baseline on the canonical `MachineFeatureVector` produced by `02_feature_engineering.ipynb` and evaluate it on a held-out **recording/load-aware** test split.

Split policy (see `ml/README.md` for details):

| split      | motor loads | recordings/class |
|------------|-------------|------------------|
| train      | 0 HP, 1 HP  | 2                |
| validation | 2 HP        | 1                |
| test       | 3 HP        | 1                |

50 / 25 / 25 at the recording level. This is the same protocol executed non-interactively by `python -m ml.src.run_experiment`; the notebook exposes the same steps for interactive inspection and does **not** hard-code any metric — everything below is computed at run time from the parquet feature file.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'ml').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'ml' / 'models'
FEATURES_PARQUET = PROCESSED_DIR / 'cwru_features.parquet'
FEATURES_CSV = PROCESSED_DIR / 'cwru_features.csv'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd

if FEATURES_CSV.is_file():
    features_df = pd.read_csv(FEATURES_CSV)
elif FEATURES_PARQUET.is_file():
    features_df = pd.read_parquet(FEATURES_PARQUET)
else:
    raise SystemExit(
        f'Neither {FEATURES_CSV} nor {FEATURES_PARQUET} exists. '
        'Run 02_feature_engineering.ipynb or `python -m ml.src.run_experiment` first.'
    )
features_df.head()

In [ ]:
from ml.src.split_dataset import load_aware_split, summarize_split

split = load_aware_split(
    features_df,
    train_loads=(0.0, 1.0),
    validation_loads=(2.0,),
    test_loads=(3.0,),
)
summarize_split(split)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from ml.src.train_baseline import FEATURE_COLUMNS, LABEL_COLUMN

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
)
clf.fit(split.train[list(FEATURE_COLUMNS)], split.train[LABEL_COLUMN])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

CLASS_ORDER = ['NORMAL', 'INNER_RACE', 'BALL', 'OUTER_RACE']
y_val = split.validation[LABEL_COLUMN]
y_val_pred = clf.predict(split.validation[list(FEATURE_COLUMNS)])
print('VALIDATION (2 HP)')
print(f'  accuracy         : {accuracy_score(y_val, y_val_pred):.4f}')
print(f'  macro precision  : {precision_score(y_val, y_val_pred, average="macro", zero_division=0):.4f}')
print(f'  macro recall     : {recall_score(y_val, y_val_pred, average="macro", zero_division=0):.4f}')
print(f'  macro F1         : {f1_score(y_val, y_val_pred, average="macro", zero_division=0):.4f}')
print()
print(classification_report(y_val, y_val_pred, labels=CLASS_ORDER, digits=4, zero_division=0))
pd.DataFrame(
    confusion_matrix(y_val, y_val_pred, labels=CLASS_ORDER),
    index=CLASS_ORDER,
    columns=CLASS_ORDER,
)

In [ ]:
y_test = split.test[LABEL_COLUMN]
y_test_pred = clf.predict(split.test[list(FEATURE_COLUMNS)])
print('TEST (3 HP) - single final evaluation')
print(f'  accuracy         : {accuracy_score(y_test, y_test_pred):.4f}')
print(f'  macro precision  : {precision_score(y_test, y_test_pred, average="macro", zero_division=0):.4f}')
print(f'  macro recall     : {recall_score(y_test, y_test_pred, average="macro", zero_division=0):.4f}')
print(f'  macro F1         : {f1_score(y_test, y_test_pred, average="macro", zero_division=0):.4f}')
print()
print(classification_report(y_test, y_test_pred, labels=CLASS_ORDER, digits=4, zero_division=0))
pd.DataFrame(
    confusion_matrix(y_test, y_test_pred, labels=CLASS_ORDER),
    index=CLASS_ORDER,
    columns=CLASS_ORDER,
)

In [ ]:
importance = pd.Series(clf.feature_importances_, index=list(FEATURE_COLUMNS)).sort_values(ascending=False)
importance